In [1]:
# Assignment 2 - Environment check (versions)

import sys
import torch
import transformers
import datasets
import evaluate
import accelerate

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Evaluate:", evaluate.__version__)
print("Accelerate:", accelerate.__version__)
print("CUDA available:", torch.cuda.is_available())

Python: 3.10.11
Torch: 2.10.0+cpu
Transformers: 5.1.0
Datasets: 4.5.0
Evaluate: 0.4.6
Accelerate: 1.12.0
CUDA available: False


In [2]:
# Load IMDb dataset and basic inspection

from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset)
print("Train size:", len(dataset["train"]))
print("Test size:", len(dataset["test"]))
print("\nExample item:")
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
Train size: 25000
Test size: 25000

Example item:
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in th

In [3]:
# DistilBERT tokenizer setup

from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Test tokenization on one example
example = dataset["train"][0]["text"]
tokens = tokenizer(example, truncation=True)

print("Number of input IDs:", len(tokens["input_ids"]))
print("First 10 token IDs:", tokens["input_ids"][:10])

Number of input IDs: 363
First 10 token IDs: [101, 1045, 12524, 1045, 2572, 8025, 1011, 3756, 2013, 2026]


In [4]:
# Tokenize entire dataset

def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

print(tokenized_dataset)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})


In [5]:
# Load DistilBERT for sequence classification

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

print("Model loaded successfully.")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully.


In [6]:
# Metrics: Accuracy and F1

import numpy as np
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1}

print("Metrics ready.")

Metrics ready.


In [7]:
# Create train/validation subsets for CPU training

from datasets import DatasetDict

small_train = tokenized_dataset["train"].shuffle(seed=42).select(range(2000))
small_test = tokenized_dataset["test"].shuffle(seed=42).select(range(1000))

small_data = DatasetDict({"train": small_train, "test": small_test})

print("Small train size:", len(small_data["train"]))
print("Small test size:", len(small_data["test"]))

Small train size: 2000
Small test size: 1000


In [8]:
# Fine-tuning configuration (CPU-friendly)

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="../results/a2_distilbert_imdb",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_data["train"],
    eval_dataset=small_data["test"],
    compute_metrics=compute_metrics,
)

print("Trainer ready.")

Trainer ready.


In [9]:
# Train and evaluate DistilBERT on IMDb subset

train_result = trainer.train()
eval_result = trainer.evaluate()

print("Eval results:", eval_result)

e:\JYU\NLP\NLP\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.308703,0.331422,0.863000,0.863003


e:\JYU\NLP\NLP\venv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Eval results: {'eval_loss': 0.33142176270484924, 'eval_accuracy': 0.863, 'eval_f1': 0.8630031526677613, 'eval_runtime': 353.6869, 'eval_samples_per_second': 2.827, 'eval_steps_per_second': 0.353, 'epoch': 1.0}
